In [0]:
BASE_PATH = "/Volumes/workspace/default/e-commerce_recommendation/"

In [0]:
orders_df = spark.read.csv(BASE_PATH + "orders.csv",
                           header=True,
                           inferSchema=True)

products_df = spark.read.csv(BASE_PATH + "products.csv",
                             header=True,
                             inferSchema=True)

prior_df = spark.read.csv(BASE_PATH + "order_products__prior.csv",
                          header=True,
                          inferSchema=True)

train_df = spark.read.csv(BASE_PATH + "order_products__train.csv",
                          header=True,
                          inferSchema=True)

aisles_df = spark.read.csv(BASE_PATH + "aisles.csv",
                           header=True,
                           inferSchema=True)

departments_df = spark.read.csv(BASE_PATH + "departments.csv",
                                header=True,
                                inferSchema=True)

In [0]:
from pyspark.sql.functions import col

products_df = (
    products_df
    .withColumn("aisle_id", col("aisle_id").cast("int"))
    .withColumn("department_id", col("department_id").cast("int"))
)

products_df.printSchema()

In [0]:
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("quote", '"')
    .csv(BASE_PATH + "products.csv")
)

In [0]:
products_df.printSchema()

In [0]:
products_df.filter(
    products_df.product_id == 6816
).show(truncate=False)

# Save Bronze Tables

In [0]:
orders_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_orders")

products_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_products")

prior_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_prior")

train_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_train")

aisles_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_aisles")

departments_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_departments")

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
raw = spark.read.text(BASE_PATH + "products.csv")

display(raw.limit(20))

## # Silver Tables

In [0]:
silver_df = (
    prior_df
    .join(orders_df, on='order_id', how='inner')
)

In [0]:
display(silver_df.limit(5))

In [0]:
silver_df = (
    silver_df
    .join(products_df, on='product_id', how='inner')
)

In [0]:
silver_df = (
    silver_df
    .join(aisles_df, on='aisle_id', how='inner')
)

In [0]:
silver_df = (
    silver_df
    .join(departments_df, on='department_id', how='inner')
)

In [0]:
display(silver_df.limit(5))

In [0]:
silver_df.printSchema()

In [0]:
products_df.filter(
    (~col("aisle_id").rlike("^[0-9]+$")) |
    (~col("department_id").rlike("^[0-9]+$"))
).show(20, truncate=False)

In [0]:
silver_df.limit(5).show(truncate=False)

In [0]:
silver_df.count()

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customer_products")

## GOLD LAYER

In [0]:
from pyspark.sql.functions import count

gold_df = (
    silver_df
    .groupBy("user_id","product_id")
    .agg(
        count("*").alias("rating")
    )
)

In [0]:
test_df = silver_df.limit(1000)

gold_test = (
    test_df
    .groupBy("user_id","product_id")
    .agg(count("*").alias("rating"))
)

display(gold_test)

In [0]:
gold_df.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("gold_interaction_matrix")